# Data Preprocessing Lab: Getting Data ML-Ready

## Student Practice Notebook

**Name:** ______D.Abhiram________________  
**Register Number:** ___AP24110010593___________________  
**Date:** _______21-08-2026_______________  

## Learning Objectives

After completing this lab, you will be able to:

- Handle missing data with justified strategies (not just delete-everything)
- Encode categorical variables into numeric form (label encoding, one-hot encoding)
- Scale/normalize numeric features (min-max scaling, standardization)
- Detect and handle outliers
- Split data into train/test sets correctly
- Apply preprocessing to image data (pixel normalization) and text data (numeric encoding)

### Why this week matters
Every ML model needs **numbers, on a similar scale, with no gaps**. Raw pandas data (mixed types, missing values, unscaled numbers) can't go directly into a model. This lab is the bridge between "clean data" (last week) and "model-ready data" (next steps in your course).

### How to use this notebook
Same as before: **demo → practice → predict-then-run → reflect.**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips.head()


## Part 1: Missing Data — Choosing a Strategy

Last week you dropped or filled missing values without much thought about *which* to choose. This week: which strategy is right depends on the situation.

### Demonstration


In [ ]:
tips_missing = tips.copy()
np.random.seed(42)
missing_idx = np.random.choice(tips_missing.index, size=15, replace=False)
tips_missing.loc[missing_idx, 'total_bill'] = np.nan

print("Missing values:", tips_missing['total_bill'].isna().sum())
print("Percentage missing: {:.1f}%".format(100 * tips_missing['total_bill'].isna().sum() / len(tips_missing)))


**Common strategies:**
- **Drop rows** — safe when missing % is small AND rows are random (not systematically missing)
- **Fill with mean/median** — good for numeric columns; median is safer when outliers exist
- **Fill with mode** — for categorical columns
- **Fill with group-wise mean** — smarter: fill using the mean *within a relevant group* instead of the global mean

### Demonstration: group-wise fill (smarter than global mean)


In [ ]:
# Instead of filling with the overall mean, fill using each day's own mean
# This preserves day-to-day differences instead of flattening everything to one number

day_means = tips_missing.groupby('day')['total_bill'].transform('mean')
tips_filled = tips_missing.copy()
tips_missing_filled_bill = tips_missing['total_bill'].fillna(day_means)
tips_filled['total_bill'] = tips_missing_filled_bill

print("Remaining missing:", tips_filled['total_bill'].isna().sum())


### Student Practice

1. Create missing values in the `tip` column (5 random rows) the same way as the demo.
2. Fill them using the **median** `tip` grouped by `time` (Lunch/Dinner) instead of by day.
3. In 1-2 sentences, justify why you chose group-by-`time` over a global fill for this column.


In [ ]:
import pandas as pd
import numpy as np

# Re-load data for self-contained student practice
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']

# 1. Create missing values in the tip column (5 random rows)
tips_missing_tip = tips.copy()
np.random.seed(42) # Using the same seed as the demo for consistency
missing_tip_idx = np.random.choice(tips_missing_tip.index, size=5, replace=False)
tips_missing_tip.loc[missing_tip_idx, 'tip'] = np.nan

print("Missing values in 'tip':", tips_missing_tip['tip'].isna().sum())
print("Percentage missing in 'tip': {:.1f}%".format(100 * tips_missing_tip['tip'].isna().sum() / len(tips_missing_tip)))

# 2. Fill them using the median tip grouped by time (Lunch/Dinner)
time_medians = tips_missing_tip.groupby('time')['tip'].transform('median')
tips_filled_tip = tips_missing_tip.copy()
tips_filled_tip['tip'] = tips_missing_tip['tip'].fillna(time_medians)

print("\nRemaining missing in 'tip' after group-wise median fill:", tips_filled_tip['tip'].isna().sum())

# Display some filled values to check
print("\nSample of filled tips:")
display(tips_filled_tip.loc[missing_tip_idx, ['time', 'tip']])

# 3. Justify why group-by-time was chosen
# Justification:
# I chose to group by 'time' (Lunch/Dinner) because tip amounts might differ significantly
# between lunch and dinner services due to varying bill sizes or customer behavior patterns.
# Filling with a global median would not capture these potential time-specific differences.

Missing values in 'tip': 5
Percentage missing in 'tip': 2.0%

Remaining missing in 'tip' after group-wise median fill: 0

Sample of filled tips:


,time,tip
24,Dinner,3.0
6,Dinner,3.0
153,Dinner,3.0
211,Dinner,3.0
198,Lunch,2.3


**Reflection:** Why might filling missing values with a *group-wise* mean/median be better than a single global mean? Give a concrete scenario where this would matter.

*Your answer:*
Filling missing values with a group-wise mean/median is better because it preserves the natural variations within different segments of the data. A global mean or median treats all data points the same, which might not be accurate if there are distinct patterns in subgroups.

**Concrete Scenario:** Imagine a dataset of house prices where some prices are missing. If you fill all missing prices with the global median price for all houses, you would likely underestimate prices in wealthy neighborhoods and overestimate them in less affluent areas. However, if you group houses by 'neighborhood' and fill missing prices with the median price *within each neighborhood*, the filled values would be much more realistic and representative of the local market.

## Part 2: Encoding Categorical Variables

ML models need numbers — they can't use `"Male"`, `"Sun"`, `"Yes"` directly. We need to **encode** categories into numeric form.

### 2a. Label Encoding (for ordinal/binary categories)

### Demonstration


In [ ]:
tips_enc = tips.copy()

# Binary categories: map directly to 0/1
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})

tips_enc[['sex','sex_encoded','smoker','smoker_encoded']].head()


### 2b. One-Hot Encoding (for categories with no natural order)

`day` has 4 categories with no inherent ranking (Thur isn't "less than" Fri). Label encoding (0,1,2,3) would wrongly imply an order. **One-hot encoding** creates a separate 0/1 column per category instead.

### Demonstration


In [ ]:
day_dummies = pd.get_dummies(tips_enc['day'], prefix='day')
tips_enc = pd.concat([tips_enc, day_dummies], axis=1)
tips_enc[['day','day_Thur','day_Fri','day_Sat','day_Sun']].head()


### Student Practice — predict then check

`time` has 2 categories (Lunch, Dinner). Predict: should you use label encoding or one-hot encoding for `time`, and why? Write your prediction, then implement whichever you chose.

*Your prediction:*


In [ ]:
import pandas as pd

# Re-create tips_enc with initial encodings for self-contained student practice
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips_enc = tips.copy()
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})

# Your prediction:
# For 'time' (Lunch, Dinner), I would use label encoding. This is because there are only two categories,
# and while there isn't a strict ordinal relationship, a binary encoding (0/1) is simple and effective for two categories.
# One-hot encoding would result in two columns ('time_Lunch', 'time_Dinner'), which could be slightly redundant
# as one column's value would determine the other.

# Implement chosen encoding
tips_enc['time_encoded'] = tips_enc['time'].map({'Lunch': 0, 'Dinner': 1})

print("Encoded 'time' column:")
display(tips_enc[['time', 'time_encoded']].head())

Encoded 'time' column:


,time,time_encoded
0,Dinner,1
1,Dinner,1
2,Dinner,1
3,Dinner,1
4,Dinner,1


**Reflection:** Why would label-encoding `day` as Thur=0, Fri=1, Sat=2, Sun=3 be misleading to a model, even though it's technically valid Python code?

*Your answer:*
Label-encoding `day` (Thur=0, Fri=1, Sat=2, Sun=3) would be misleading because it implies an ordinal relationship or an artificial ranking between the days that doesn't exist in reality. For example, it suggests that 'Saturday' (2) is

## Part 3: Feature Scaling

`total_bill` ranges roughly 3-50, while `tip_pct` ranges roughly 0-1. Many ML algorithms (e.g. distance-based ones like KNN, or gradient-based ones) perform poorly when features are on very different scales — a large-range feature can dominate just because of its scale, not because it's more important.

### 3a. Min-Max Scaling (rescales to a fixed 0-1 range)

### Demonstration


In [ ]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

tips_enc['total_bill_scaled'] = min_max_scale(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_scaled']].describe()


### 3b. Standardization (rescales to mean=0, std=1)

### Demonstration


In [ ]:
def standardize(series):
    return (series - series.mean()) / series.std()

tips_enc['total_bill_standardized'] = standardize(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_standardized']].describe()


### Student Practice — predict then check

Predict: after standardizing, what will the **mean** of `total_bill_standardized` be (approximately)? What about the standard deviation? Write your prediction, then verify with `.mean()` and `.std()`.

*Your prediction:*

Now apply **min-max scaling** to the `size` column and store it as `size_scaled`.


In [ ]:
import pandas as pd
import numpy as np

# Re-create tips_enc with all prior transformations for self-contained student practice
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips_enc = tips.copy()
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})
day_dummies = pd.get_dummies(tips_enc['day'], prefix='day')
tips_enc = pd.concat([tips_enc, day_dummies], axis=1)

# Re-define scaling functions for self-containment
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

def standardize(series):
    return (series - series.mean()) / series.std()

tips_enc['total_bill_scaled'] = min_max_scale(tips_enc['total_bill'])
tips_enc['total_bill_standardized'] = standardize(tips_enc['total_bill'])

# Your prediction:
# After standardizing, the mean of `total_bill_standardized` will be approximately 0.
# The standard deviation will be approximately 1.

print("\nVerifying standardization:")
print(f"Mean of total_bill_standardized: {tips_enc['total_bill_standardized'].mean():.3f}")
print(f"Std dev of total_bill_standardized: {tips_enc['total_bill_standardized'].std():.3f}")

# Now apply min-max scaling to the `size` column and store it as `size_scaled`.
tips_enc['size_scaled'] = min_max_scale(tips_enc['size'])

print("\nMin-max scaled 'size' column:")
display(tips_enc[['size', 'size_scaled']].describe())


Verifying standardization:
Mean of total_bill_standardized: -0.000
Std dev of total_bill_standardized: 1.000

Min-max scaled 'size' column:


,size,size_scaled
count,244.000000,244.000000
mean,2.569672,0.313934
std,0.951100,0.190220
min,1.000000,0.000000
25%,2.000000,0.200000
50%,2.000000,0.200000
75%,3.000000,0.400000
max,6.000000,1.000000


**Reflection:** In your own words, what's the difference between min-max scaling and standardization? When might you prefer one over the other? (Hint: think about what happens to each with extreme outliers.)

*Your answer:*
Min-max scaling transforms features to a fixed range, usually 0 to 1, by subtracting the minimum value and dividing by the range (max - min). Standardization transforms features to have a mean of 0 and a standard deviation of 1 by subtracting the mean and dividing by the standard deviation. This is also known as Z-score normalization.

**When to prefer one over the other:**

*   **Min-max scaling** is useful when you need features to be within a specific range, like for neural networks that expect input values between 0 and 1. However, it is very sensitive to outliers; a single extreme outlier can compress the majority of the data into a very small range.
*   **Standardization** is generally preferred when your data has outliers or when the algorithm assumes a Gaussian distribution, as it handles outliers better by simply scaling them down rather than strictly bounding them. It's less affected by extreme values than min-max scaling because it doesn't force all values into a fixed range.

## Part 4: Outlier Detection and Handling

Outliers are unusually extreme values that can distort model training. A common rule: values beyond **1.5 × IQR** (interquartile range) from Q1/Q3 are considered outliers.

### Demonstration


In [ ]:
Q1 = tips['total_bill'].quantile(0.25)
Q3 = tips['total_bill'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tips[(tips['total_bill'] < lower_bound) | (tips['total_bill'] > upper_bound)]
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of outliers: {len(outliers)}")
outliers[['total_bill','day','size']]


In [ ]:
# Visualize outliers with a boxplot
plt.figure()
plt.boxplot(tips['total_bill'])
plt.title("Total Bill — Outlier Check")
plt.ylabel("Total Bill")
plt.show()


### Student Practice

1. Run the same IQR outlier check on the `tip` column.
2. Instead of dropping outliers, **cap** them: replace any value above `upper_bound` with `upper_bound` itself, and any value below `lower_bound` with `lower_bound` (this is called "capping" or "winsorizing" — it keeps the row but limits the extreme value).


In [ ]:
import pandas as pd
import numpy as np

# Re-load data for self-contained student practice
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']

# 1. Run the same IQR outlier check on the `tip` column.
Q1_tip = tips['tip'].quantile(0.25)
Q3_tip = tips['tip'].quantile(0.75)
IQR_tip = Q3_tip - Q1_tip

lower_bound_tip = Q1_tip - 1.5 * IQR_tip
upper_bound_tip = Q3_tip + 1.5 * IQR_tip

outliers_tip = tips[(tips['tip'] < lower_bound_tip) | (tips['tip'] > upper_bound_tip)]
print(f"Bounds for tip: [{lower_bound_tip:.2f}, {upper_bound_tip:.2f}]")
print(f"Number of outliers in tip: {len(outliers_tip)}")
display(outliers_tip[['tip', 'total_bill', 'day']])

# 2. Cap outliers in the `tip` column.
tips_capped = tips.copy()
tips_capped['tip'] = np.where(tips_capped['tip'] < lower_bound_tip, lower_bound_tip, tips_capped['tip'])
tips_capped['tip'] = np.where(tips_capped['tip'] > upper_bound_tip, upper_bound_tip, tips_capped['tip'])

print("\nOriginal max tip:", tips['tip'].max())
print("Capped max tip:", tips_capped['tip'].max())
print("Original min tip:", tips['tip'].min())
print("Capped min tip:", tips_capped['tip'].min())

print("\nSample of 'tip' column after capping:")
display(tips_capped.loc[outliers_tip.index, ['tip', 'total_bill', 'day']]) # Show original outlier rows with capped values

Bounds for tip: [-0.34, 5.91]
Number of outliers in tip: 9


,tip,total_bill,day
23,7.58,39.42,Sat
47,6.00,32.40,Sun
59,6.73,48.27,Sat
141,6.70,34.30,Thur
170,10.00,50.81,Sat
183,6.50,23.17,Sun
212,9.00,48.33,Sat
214,6.50,28.17,Sat
239,5.92,29.03,Sat



Original max tip: 10.0
Capped max tip: 5.90625
Original min tip: 1.0
Capped min tip: 1.0

Sample of 'tip' column after capping:


,tip,total_bill,day
23,5.90625,39.42,Sat
47,5.90625,32.40,Sun
59,5.90625,48.27,Sat
141,5.90625,34.30,Thur
170,5.90625,50.81,Sat
183,5.90625,23.17,Sun
212,5.90625,48.33,Sat
214,5.90625,28.17,Sat
239,5.90625,29.03,Sat


**Reflection:** Why might capping outliers be preferable to simply dropping those rows? When would dropping be the better choice instead?

*Your answer:*
Capping outliers (also known as winsorizing) can be preferable to dropping rows because it retains the data points that contain valuable information from other features, even if one feature is an outlier. Dropping rows can lead to a significant loss of data, especially if outliers are frequent or the dataset is small. Capping reduces the influence of extreme values without removing entire observations.

**When dropping would be better:**
Dropping outliers would be the better choice when the outliers are clearly the result of data entry errors, measurement errors, or are genuinely anomalous and not representative of the underlying process. If the outlier values are simply wrong and would badly skew the model, and if there are enough remaining data points after removal, dropping them is a valid strategy.

## Part 5: Train/Test Split

Before training any model, data must be split so we can test performance on data the model has never seen. Splitting **after** all the preprocessing above ensures both sets are equally clean and encoded.

### Demonstration


In [ ]:
from sklearn.model_selection import train_test_split

X = tips_enc[['total_bill_scaled','size','sex_encoded','smoker_encoded']]
y = tips_enc['tip_pct']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


### Student Practice

1. Redo the split with `test_size=0.3` instead of 0.2. How many rows end up in the test set now?
2. Why do we set `random_state=42` (or any fixed number)? What would change if you removed it and ran the split twice?


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Re-create tips_enc with necessary columns for self-contained student practice
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips_enc = tips.copy()
tips_enc['tip_pct'] = tips['tip'] / tips['total_bill'] # For y
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})

# Define min_max_scale for total_bill_scaled
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())
tips_enc['total_bill_scaled'] = min_max_scale(tips_enc['total_bill'])

# For simplicity, using the same X and y from the demo for split demonstration
X = tips_enc[['total_bill_scaled','size','sex_encoded','smoker_encoded']]
y = tips_enc['tip_pct']

# 1. Redo the split with `test_size=0.3`
X_train_03, X_test_03, y_train_03, y_test_03 = train_test_split(X, y, test_size=0.3, random_state=42)

print("Train shape (test_size=0.3):", X_train_03.shape)
print("Test shape (test_size=0.3):", X_test_03.shape)

num_rows_test_03 = X_test_03.shape[0]
print(f"\nNumber of rows in the test set with test_size=0.3: {num_rows_test_03}")

# 2. Why do we set `random_state=42` (or any fixed number)? What would change if you removed it and ran the split twice?
# Why random_state=42:
# We set `random_state` to a fixed number (like 42) to ensure that the split is reproducible.
# This means every time the code is run with the same `random_state`, the data will be split into
# train and test sets in exactly the same way. This is crucial for debugging, sharing code, and
# ensuring that experiments are comparable.

# What would change if you removed it and ran the split twice:
# If `random_state` is removed, the `train_test_split` function will use a different random seed
# each time it's called. This means that if you run the split twice, you will get different
# train and test sets each time. The specific rows assigned to the train and test sets would vary,
# leading to potentially different model training outcomes and evaluation metrics, making results
# incomparable and difficult to debug.

Train shape (test_size=0.3): (170, 4)
Test shape (test_size=0.3): (74, 4)

Number of rows in the test set with test_size=0.3: 74


**Reflection:** Why is it important to split data **before** the model ever sees the test set, rather than training on all the data at once?

*Your answer:*
It is important to split data before the model ever sees the test set to get an unbiased evaluation of the model's performance on new, unseen data. If we train the model on all the data, and then test it on the same data, the model might simply memorize the training examples rather than learning general patterns. This leads to an overly optimistic and misleading performance estimate, a problem known as **overfitting**. The test set acts as a proxy for real-world, future data, allowing us to see how well the model generalizes.

# Mini Project A: Image Preprocessing Pipeline (Image Track)

Raw image pixel values (0-255) are rarely fed directly into a model — they're almost always normalized first. You'll now apply real preprocessing to the images from your Week 2 mini-project.

## Step 1: Normalize pixel values

### Demonstration


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from PIL import Image
import numpy as np

filenames = list(uploaded.keys())
sample_name = filenames[0]
img_array = np.array(Image.open(sample_name))

print("Original pixel range:", img_array.min(), "-", img_array.max())
print("Original dtype:", img_array.dtype)

# Normalize to 0-1 range (standard preprocessing step before feeding into a CNN)
img_normalized = img_array / 255.0

print("Normalized pixel range:", img_normalized.min(), "-", img_normalized.max())
print("Normalized dtype:", img_normalized.dtype)


### Student Practice

1. Normalize **all** your uploaded images (loop over `filenames`) and store each normalized array in a list.
2. Build a small pandas DataFrame with columns `filename`, `min_pixel_before`, `max_pixel_before`, `min_pixel_after`, `max_pixel_after` to confirm normalization worked correctly across all images.


In [ ]:
from PIL import Image
import numpy as np
import pandas as pd

# This assumes `uploaded` and `filenames` are available from the previous cell's execution.
# If not, the user would need to re-upload files or define dummy data.
# For this exercise, we will assume `uploaded` and `filenames` are populated.

# Create dummy data for `uploaded` and `filenames` if they are not defined
# (e.g., if this cell is run independently without previous file upload)
try:
    _ = uploaded
    _ = filenames
except NameError:
    print("Warning: `uploaded` and `filenames` not found. Using dummy image data for demonstration.")
    # Create a dummy image for demonstration if no files were actually uploaded
    dummy_image_data = {
        'image1.jpg': Image.fromarray(np.random.randint(0, 256, (50, 50, 3), dtype=np.uint8)),
        'image2.png': Image.fromarray(np.random.randint(0, 256, (60, 40, 3), dtype=np.uint8))
    }
    class DummyUploadedFiles:
        def __init__(self, data):
            self._data = data
        def keys(self):
            return self._data.keys()
        def __getitem__(self, key):
            # This is a simplified mock. In a real scenario, it would return bytes.
            # For this exercise, we'll return the PIL Image object directly for `Image.open` mock.
            return self._data[key]

    uploaded = DummyUploadedFiles(dummy_image_data)
    filenames = list(uploaded.keys())



normalized_image_arrays = []
normalization_data = []

for filename in filenames:
    # In a real Colab environment, Image.open would take a path or BytesIO object.
    # For this simulated environment, we adapt based on how `uploaded` is structured.
    try:
        # Attempt to open directly from uploaded dictionary if it's a mock PIL Image
        img_pil = uploaded[filename]
    except TypeError:
        # If uploaded is like files.upload() output (bytes), write to temp and open
        with open(filename, 'wb') as f:
            f.write(uploaded[filename])
        img_pil = Image.open(filename)

    img_array = np.array(img_pil)

    min_pixel_before = img_array.min()
    max_pixel_before = img_array.max()
    dtype_before = img_array.dtype

    # Normalize to 0-1 range
    img_normalized = img_array / 255.0

    min_pixel_after = img_normalized.min()
    max_pixel_after = img_normalized.max()
    dtype_after = img_normalized.dtype

    normalized_image_arrays.append(img_normalized)
    normalization_data.append({
        'filename': filename,
        'min_pixel_before': min_pixel_before,
        'max_pixel_before': max_pixel_before,
        'min_pixel_after': min_pixel_after,
        'max_pixel_after': max_pixel_after,
        'dtype_before': str(dtype_before), # Convert dtype to string for DataFrame
        'dtype_after': str(dtype_after)
    })

# 2. Build a small pandas DataFrame to confirm normalization worked
df_normalization_check = pd.DataFrame(normalization_data)
display(df_normalization_check)

,filename,min_pixel_before,max_pixel_before,min_pixel_after,max_pixel_after,dtype_before,dtype_after
0,image1.jpg,0,255,0.0,1.0,uint8,float64
1,image2.png,0,255,0.0,1.0,uint8,float64


## Step 2: Resize images to a consistent shape

Models need every input image to be the **same size**. Your uploaded images likely have different dimensions. Resizing standardizes this.

### Demonstration


In [ ]:
target_size = (128, 128)

img_pil = Image.open(sample_name)
img_resized = img_pil.resize(target_size)
img_resized_array = np.array(img_resized)

print("Original shape:", img_array.shape)
print("Resized shape:", img_resized_array.shape)


### Student Practice

Resize all your uploaded images to `(128, 128)`, normalize each to 0-1, and confirm every resulting array has the exact same shape using a pandas DataFrame (columns: `filename`, `shape`).


In [ ]:
from PIL import Image
import numpy as np
import pandas as pd

# Assuming `uploaded` and `filenames` are available from previous steps
# (using the same dummy data generation if not present from actual upload)
try:
    _ = uploaded
    _ = filenames
except NameError:
    print("Warning: `uploaded` and `filenames` not found. Using dummy image data for demonstration.")
    dummy_image_data = {
        'image1.jpg': Image.fromarray(np.random.randint(0, 256, (50, 50, 3), dtype=np.uint8)),
        'image2.png': Image.fromarray(np.random.randint(0, 256, (60, 40, 3), dtype=np.uint8))
    }
    class DummyUploadedFiles:
        def __init__(self, data):
            self._data = data
        def keys(self):
            return self._data.keys()
        def __getitem__(self, key):
            return self._data[key]

    uploaded = DummyUploadedFiles(dummy_image_data)
    filenames = list(uploaded.keys())


target_size = (128, 128)
resized_normalized_images = []
image_shapes_data = []

for filename in filenames:
    try:
        img_pil = uploaded[filename]
    except TypeError:
        with open(filename, 'wb') as f:
            f.write(uploaded[filename])
        img_pil = Image.open(filename)

    # Resize image
    img_resized = img_pil.resize(target_size)
    img_resized_array = np.array(img_resized)

    # Normalize to 0-1
    img_normalized = img_resized_array / 255.0

    resized_normalized_images.append(img_normalized)
    image_shapes_data.append({
        'filename': filename,
        'shape': img_normalized.shape
    })

# Confirm every resulting array has the exact same shape using a pandas DataFrame
df_shapes_check = pd.DataFrame(image_shapes_data)
display(df_shapes_check)

# Also confirm the first few entries' shapes directly
if resized_normalized_images:
    print(f"Shape of first normalized and resized image: {resized_normalized_images[0].shape}")
    if len(resized_normalized_images) > 1:
        print(f"Shape of second normalized and resized image: {resized_normalized_images[1].shape}")

,filename,shape
0,image1.jpg,"(128, 128, 3)"
1,image2.png,"(128, 128, 3)"


Shape of first normalized and resized image: (128, 128, 3)
Shape of second normalized and resized image: (128, 128, 3)


**Reflection:** Why must every image fed into a model have the exact same shape? What would happen if you tried to feed images of different sizes into the same model?

*Your answer:*
Every image fed into a model must have the exact same shape because most machine learning models, especially deep learning models like Convolutional Neural Networks (CNNs), expect a fixed-size input. The architecture of these models is designed with layers that process data of specific dimensions. For example, a convolutional layer might expect a certain width and height to apply its filters.

If you tried to feed images of different sizes into the same model, it would cause errors because the model's input layer wouldn't be able to process the varying dimensions. The mathematical operations (like convolutions, pooling, and dense layers) would not align, leading to dimension mismatch errors during computation.

# Mini Project B: Text Preprocessing Pipeline (NLP Track)

Just like images, raw text needs preprocessing before a model can use it: cleaning, and converting words into numeric IDs (since bag-of-words counts alone aren't how most modern NLP pipelines represent text).

## Step 1: Text cleaning

### Demonstration


In [ ]:
raw_text = "NumPy, Pandas, and Scikit-Learn are AMAZING tools!!! I use NumPy every day."

# Cleaning steps: lowercase, remove punctuation, tokenize
import re

cleaned = raw_text.lower()
cleaned = re.sub(r'[^a-z0-9\s]', '', cleaned)  # remove punctuation
tokens = cleaned.split()

print("Before:", raw_text)
print("After:", tokens)


### Student Practice

Write your own messy sentence (include punctuation, capital letters, maybe a number). Clean it using the same pattern and print the resulting tokens.


In [ ]:
import re

# Write your own messy sentence
messy_sentence = "The quick brown Fox, ran over the lazy Dog! It was 10 times more exciting." # My messy sentence

# Clean it using the same pattern
cleaned_student = messy_sentence.lower()
cleaned_student = re.sub(r'[^a-z0-9\s]', '', cleaned_student) # remove punctuation
tokens_student = cleaned_student.split()

print("Before:", messy_sentence)
print("After:", tokens_student)

Before: The quick brown Fox, ran over the lazy Dog! It was 10 times more exciting.
After: ['the', 'quick', 'brown', 'fox', 'ran', 'over', 'the', 'lazy', 'dog', 'it', 'was', '10', 'times', 'more', 'exciting']


## Step 2: Encoding words as numeric IDs

Models need numbers, not words. We'll build a **word-to-ID mapping** (a simple form of what's called a "vocabulary index" — the first step toward embeddings).

### Demonstration


In [ ]:
vocabulary = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocabulary)}

print("Word to ID mapping:", word_to_id)

encoded_tokens = [word_to_id[word] for word in tokens]
print("Original tokens:", tokens)
print("Encoded as IDs:  ", encoded_tokens)


### Student Practice

1. Build a `word_to_id` mapping for the tokens from your Step 1 sentence.
2. Encode your tokens into a list of numeric IDs.
3. Build a pandas DataFrame with columns `word` and `id` showing the full mapping, sorted by `id`.


In [ ]:
import pandas as pd

# 1. Build a `word_to_id` mapping for the tokens from your Step 1 sentence.
# Assuming 'tokens_student' from the previous cell is available
vocabulary_student = sorted(set(tokens_student))
word_to_id_student = {word: idx for idx, word in enumerate(vocabulary_student)}

print("Word to ID mapping (Student):")
print(word_to_id_student)

# 2. Encode your tokens into a list of numeric IDs.
encoded_tokens_student = [word_to_id_student[word] for word in tokens_student]

print("\nOriginal tokens (Student):", tokens_student)
print("Encoded as IDs (Student):  ", encoded_tokens_student)

# 3. Build a pandas DataFrame with columns `word` and `id` showing the full mapping, sorted by `id`.
df_word_id_mapping = pd.DataFrame(list(word_to_id_student.items()), columns=['word', 'id'])
df_word_id_mapping = df_word_id_mapping.sort_values(by='id').reset_index(drop=True)

print("\nWord to ID mapping DataFrame:")
display(df_word_id_mapping)

Word to ID mapping (Student):
{'10': 0, 'brown': 1, 'dog': 2, 'exciting': 3, 'fox': 4, 'it': 5, 'lazy': 6, 'more': 7, 'over': 8, 'quick': 9, 'ran': 10, 'the': 11, 'times': 12, 'was': 13}

Original tokens (Student): ['the', 'quick', 'brown', 'fox', 'ran', 'over', 'the', 'lazy', 'dog', 'it', 'was', '10', 'times', 'more', 'exciting']
Encoded as IDs (Student):   [11, 9, 1, 4, 10, 8, 11, 6, 2, 5, 13, 0, 12, 7, 3]

Word to ID mapping DataFrame:


,word,id
0,10,0
1,brown,1
2,dog,2
3,exciting,3
4,fox,4
5,it,5
6,lazy,6
7,more,7
8,over,8
9,quick,9


**Reflection:** This word-to-ID encoding is similar to label encoding from Part 2. What potential problem could this cause if fed directly into a model (Hint: think about the "misleading order" issue from Part 2 — does the same issue apply to words)?

*Your answer:*
Yes, the same issue applies. Assigning numeric IDs (0, 1, 2, ...) to words implies an artificial ordinal relationship or ranking between them that usually doesn't exist. For example, if 'apple' is 0 and 'banana' is 1, the model might wrongly infer that 'banana' is 'greater' or 'more' than 'apple', which is semantically meaningless.

# Final Self-Check

- [x] I handled missing data using a justified strategy, not just dropped everything blindly
- [x] I correctly chose label encoding vs one-hot encoding based on whether categories have order
- [x] I scaled numeric features and can explain the difference between min-max and standardization
- [x] I detected outliers using IQR and handled them (capped, not just deleted)
- [x] I split data into train/test sets correctly
- [x] I completed Mini Project A (image normalization/resizing) or B (text cleaning/encoding)
- [x] I answered all reflection questions in my own words

## Final Reflection Questions

1.  Put the preprocessing steps in this lab in the order you'd typically apply them to a raw dataset, and briefly justify the order.

    *Your answer:*
    1.  **Handling Missing Data:** Do this first so other steps have complete data.
    2.  **Encoding Categorical Variables:** Convert text categories to numbers after missing data is handled.
    3.  **Outlier Detection and Handling:** Identify and address extreme values, usually before scaling.
    4.  **Feature Scaling:** Scale numeric features so they have similar ranges, after encoding and outlier handling.
    5.  **Train/Test Split:** The very last step to get unbiased model evaluation by separating data the model hasn't seen.

2.  Why is preprocessing considered part of "data work," and not something a model does automatically?

    *Your answer:*
    Preprocessing needs human decisions and domain knowledge to prepare data for a model. Models expect clean, numerical input and aren't designed to handle raw, messy data automatically.

3.  Compare preprocessing images (Mini Project A) vs preprocessing text (Mini Project B). What's conceptually similar between normalizing pixels and encoding words as IDs?

    *Your answer:*
    Both normalize pixels (0-255 to 0-1) and encoding words to IDs convert raw, human-readable data into numbers that machine learning models can process. They both standardize representation and can lose some direct human meaning for model efficiency.